# ASD Prediction System - Data Exploration

This notebook provides comprehensive exploratory data analysis (EDA) for the ASD screening dataset.

## Contents
1. Data Loading and Overview
2. Missing Value Analysis
3. Target Variable Analysis
4. Feature Distributions
5. Correlation Analysis
6. Feature-Target Relationships
7. Key Insights Summary

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

# Set up paths
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))

# Import custom modules
from data_processing.data_loader import DataLoader

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print(f"Project root: {project_root}")

## 1. Data Loading and Overview

In [ ]:
# Load the training data
data_path = project_root / 'data' / 'raw'
loader = DataLoader(data_path)

# Load training data
df = loader.load_raw_data('asd_train_data.csv')

print(f"Dataset shape: {df.shape}")
print(f"Number of features: {df.shape[1] - 2}")  # Excluding ID and target
print(f"Number of samples: {df.shape[0]}")

In [ ]:
# Display first few rows
df.head(10)

In [ ]:
# Data types overview
print("Data Types:")
print("="*50)
print(df.dtypes)
print("\nData Info:")
df.info()

In [ ]:
# Statistical summary
df.describe()

In [ ]:
# Categorical columns summary
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical Columns:")
for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())

## 2. Missing Value Analysis

In [ ]:
# Calculate missing values
missing_counts = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print(f"Columns with missing values: {len(missing_df)}")
print(f"Total missing values: {df.isnull().sum().sum()}")
print(f"\nMissing Value Summary:")
missing_df

In [ ]:
# Visualize missing values
if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    missing_df['Missing %'].plot(kind='bar', ax=ax, color='coral')
    ax.set_title('Missing Values by Feature', fontsize=14)
    ax.set_xlabel('Feature')
    ax.set_ylabel('Missing Percentage (%)')
    ax.axhline(y=5, color='red', linestyle='--', label='5% threshold')
    plt.xticks(rotation=45, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No missing values in the dataset!")

## 3. Target Variable Analysis

In [ ]:
# Target distribution
target_counts = df['asd_diagnosis'].value_counts()
target_pct = df['asd_diagnosis'].value_counts(normalize=True) * 100

print("ASD Diagnosis Distribution:")
print("="*40)
print(f"No ASD (0): {target_counts[0]} ({target_pct[0]:.1f}%)")
print(f"ASD (1):    {target_counts[1]} ({target_pct[1]:.1f}%)")
print(f"\nClass Imbalance Ratio: {target_counts[0]/target_counts[1]:.2f}:1")

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar plot
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['No ASD (0)', 'ASD (1)'], target_counts.values, color=colors)
axes[0].set_title('ASD Diagnosis Distribution', fontsize=14)
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(target_counts.values, labels=['No ASD', 'ASD'], 
            autopct='%1.1f%%', colors=colors, startangle=90,
            explode=[0, 0.05])
axes[1].set_title('ASD Diagnosis Proportion', fontsize=14)

plt.tight_layout()
plt.show()

## 4. Feature Distributions

In [ ]:
# Identify feature types
behavioral_features = ['eye_contact', 'response_to_name', 'pointing', 'social_smile', 
                       'repetitive_behaviors', 'joint_attention', 'pretend_play',
                       'unusual_interests', 'hand_flapping', 'toe_walking', 
                       'lines_up_toys', 'upset_by_change']

communication_features = ['word_count', 'two_word_phrases', 'echolalia', 'language_regression']

demographic_features = ['age_months', 'gender', 'geographic_location', 'setting_type', 
                        'family_history_asd', 'birth_order', 'gestational_weeks']

screening_scores = ['mchat_score', 'social_communication_score', 'rrb_score']

# Check which features exist in our data
existing_behavioral = [f for f in behavioral_features if f in df.columns]
existing_communication = [f for f in communication_features if f in df.columns]
existing_demographic = [f for f in demographic_features if f in df.columns]
existing_screening = [f for f in screening_scores if f in df.columns]

print(f"Behavioral features: {len(existing_behavioral)}")
print(f"Communication features: {len(existing_communication)}")
print(f"Demographic features: {len(existing_demographic)}")
print(f"Screening scores: {len(existing_screening)}")

In [ ]:
# Plot behavioral feature distributions by ASD status
if existing_behavioral:
    n_cols = 4
    n_rows = (len(existing_behavioral) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
    axes = axes.flatten()
    
    for i, feature in enumerate(existing_behavioral):
        # Group by ASD diagnosis
        asd_counts = df.groupby(['asd_diagnosis', feature]).size().unstack(fill_value=0)
        asd_counts.plot(kind='bar', ax=axes[i], color=['#2ecc71', '#e74c3c'])
        axes[i].set_title(feature.replace('_', ' ').title())
        axes[i].set_xlabel('ASD Diagnosis')
        axes[i].set_xticklabels(['No ASD', 'ASD'], rotation=0)
        axes[i].legend(['0 (Typical)', '1 (Concern)'], title='Response')
    
    # Hide unused subplots
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Behavioral Features by ASD Diagnosis', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot continuous feature distributions
continuous_features = ['age_months', 'word_count', 'mchat_score', 
                       'social_communication_score', 'rrb_score', 'gestational_weeks']
existing_continuous = [f for f in continuous_features if f in df.columns]

if existing_continuous:
    n_cols = 3
    n_rows = (len(existing_continuous) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 and n_cols == 1 else axes
    
    for i, feature in enumerate(existing_continuous):
        # Histogram by ASD status
        for asd_val, color, label in [(0, '#2ecc71', 'No ASD'), (1, '#e74c3c', 'ASD')]:
            data = df[df['asd_diagnosis'] == asd_val][feature].dropna()
            axes[i].hist(data, bins=20, alpha=0.6, label=label, color=color)
        
        axes[i].set_title(feature.replace('_', ' ').title())
        axes[i].set_xlabel(feature)
        axes[i].set_ylabel('Frequency')
        axes[i].legend()
    
    # Hide unused subplots
    for j in range(len(existing_continuous), len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle('Continuous Features by ASD Diagnosis', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Demographic feature analysis
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Gender distribution by ASD
if 'gender' in df.columns:
    gender_asd = pd.crosstab(df['gender'], df['asd_diagnosis'], normalize='index') * 100
    gender_asd.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
    axes[0].set_title('ASD Rate by Gender')
    axes[0].set_ylabel('Percentage (%)')
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
    axes[0].legend(['No ASD', 'ASD'])

# Setting type distribution
if 'setting_type' in df.columns:
    setting_asd = pd.crosstab(df['setting_type'], df['asd_diagnosis'], normalize='index') * 100
    setting_asd.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
    axes[1].set_title('ASD Rate by Setting Type')
    axes[1].set_ylabel('Percentage (%)')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)
    axes[1].legend(['No ASD', 'ASD'])

# Family history
if 'family_history_asd' in df.columns:
    fam_asd = pd.crosstab(df['family_history_asd'], df['asd_diagnosis'], normalize='index') * 100
    fam_asd.plot(kind='bar', ax=axes[2], color=['#2ecc71', '#e74c3c'])
    axes[2].set_title('ASD Rate by Family History')
    axes[2].set_ylabel('Percentage (%)')
    axes[2].set_xticklabels(['No Family History', 'Family History'], rotation=0)
    axes[2].legend(['No ASD', 'ASD'])

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Select numeric columns for correlation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'participant_id']

# Calculate correlation matrix
corr_matrix = df[numeric_cols].corr()

# Plot correlation heatmap
plt.figure(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', 
            cmap='RdBu_r', center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with target
target_corr = corr_matrix['asd_diagnosis'].drop('asd_diagnosis').sort_values(key=abs, ascending=False)

print("Feature Correlations with ASD Diagnosis:")
print("="*50)
for feature, corr in target_corr.items():
    direction = "↑" if corr > 0 else "↓"
    print(f"{feature:35} {corr:+.3f} {direction}")

In [ ]:
# Visualize top correlated features
top_n = 15
top_corr = target_corr.head(top_n)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c' if x > 0 else '#3498db' for x in top_corr.values]
top_corr.plot(kind='barh', ax=ax, color=colors)
ax.set_title(f'Top {top_n} Features Correlated with ASD Diagnosis', fontsize=14)
ax.set_xlabel('Correlation Coefficient')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

## 6. Feature-Target Relationships

In [ ]:
# Box plots for continuous features by ASD status
continuous_features = ['age_months', 'word_count', 'mchat_score', 
                       'social_communication_score', 'rrb_score']
existing_cont = [f for f in continuous_features if f in df.columns]

if existing_cont:
    n_cols = 3
    n_rows = (len(existing_cont) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
    axes = axes.flatten() if n_rows > 1 else axes
    
    for i, feature in enumerate(existing_cont):
        sns.boxplot(data=df, x='asd_diagnosis', y=feature, ax=axes[i],
                    palette=['#2ecc71', '#e74c3c'])
        axes[i].set_title(f'{feature.replace("_", " ").title()} by ASD Status')
        axes[i].set_xticklabels(['No ASD', 'ASD'])
    
    for j in range(len(existing_cont), len(axes)):
        axes[j].set_visible(False)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Statistical comparison between ASD and non-ASD groups
from scipy import stats

print("Statistical Comparison: ASD vs Non-ASD Groups")
print("="*70)
print(f"{'Feature':<30} {'No ASD Mean':>12} {'ASD Mean':>12} {'p-value':>12}")
print("-"*70)

for feature in existing_cont:
    no_asd = df[df['asd_diagnosis'] == 0][feature].dropna()
    asd = df[df['asd_diagnosis'] == 1][feature].dropna()
    
    # T-test
    t_stat, p_value = stats.ttest_ind(no_asd, asd)
    
    significance = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else ""
    
    print(f"{feature:<30} {no_asd.mean():>12.2f} {asd.mean():>12.2f} {p_value:>12.4f} {significance}")

print("\n* p<0.05, ** p<0.01, *** p<0.001")

In [ ]:
# Chi-square test for binary features
binary_features = [f for f in existing_behavioral if f in df.columns]

print("\nChi-Square Test for Binary Features")
print("="*70)
print(f"{'Feature':<30} {'No ASD %':>12} {'ASD %':>12} {'Chi2':>10} {'p-value':>12}")
print("-"*70)

for feature in binary_features:
    # Create contingency table
    contingency = pd.crosstab(df[feature], df['asd_diagnosis'])
    chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
    
    # Calculate percentages of '1' (concern) in each group
    no_asd_pct = df[df['asd_diagnosis'] == 0][feature].mean() * 100
    asd_pct = df[df['asd_diagnosis'] == 1][feature].mean() * 100
    
    significance = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else ""
    
    print(f"{feature:<30} {no_asd_pct:>11.1f}% {asd_pct:>11.1f}% {chi2:>10.2f} {p_value:>12.4f} {significance}")

print("\n* p<0.05, ** p<0.01, *** p<0.001")

## 7. Key Insights Summary

In [ ]:
# Generate summary insights
print("="*70)
print("KEY INSIGHTS FROM EXPLORATORY DATA ANALYSIS")
print("="*70)

print("\n1. DATASET OVERVIEW")
print(f"   - Total samples: {len(df)}")
print(f"   - Total features: {len(df.columns) - 2}")
print(f"   - Missing values: {df.isnull().sum().sum()} ({df.isnull().sum().sum()/df.size*100:.2f}%)")

print("\n2. TARGET DISTRIBUTION")
asd_rate = df['asd_diagnosis'].mean() * 100
print(f"   - ASD prevalence in dataset: {asd_rate:.1f}%")
print(f"   - Class imbalance ratio: {(100-asd_rate)/asd_rate:.1f}:1")

print("\n3. TOP PREDICTIVE FEATURES (by correlation)")
for i, (feature, corr) in enumerate(target_corr.head(5).items(), 1):
    print(f"   {i}. {feature}: r = {corr:.3f}")

print("\n4. BEHAVIORAL FEATURE INSIGHTS")
if binary_features:
    most_discriminative = []
    for feature in binary_features:
        no_asd_rate = df[df['asd_diagnosis'] == 0][feature].mean()
        asd_rate_feat = df[df['asd_diagnosis'] == 1][feature].mean()
        diff = asd_rate_feat - no_asd_rate
        most_discriminative.append((feature, diff))
    
    most_discriminative.sort(key=lambda x: abs(x[1]), reverse=True)
    for feature, diff in most_discriminative[:5]:
        print(f"   - {feature}: {abs(diff)*100:.1f}% difference between groups")

print("\n5. RECOMMENDATIONS FOR MODELING")
print("   - Handle class imbalance (SMOTE, class weights, or threshold adjustment)")
print("   - Focus on behavioral and communication features")
print("   - Consider feature interactions (e.g., eye_contact * social_smile)")
print("   - Use stratified cross-validation due to class imbalance")
print("   - Prioritize sensitivity (recall) for screening purposes")

print("\n" + "="*70)

In [ ]:
# Save summary statistics
summary_stats = loader.get_data_summary(df)

print("\nSummary statistics saved to memory.")
print("Ready for preprocessing and model training!")